## Set Up

In [2]:
import pandas as pd
import numpy as np
import json
import logging
from typing import List, Tuple, Dict, Any, Optional
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM
from transformers import BitsAndBytesConfig
from transformers import GenerationConfig
from transformers import AutoModel, AutoTokenizer, AutoModelForCausalLM, LlamaForCausalLM, LlamaTokenizerFast
from peft import PeftModel
import warnings
import torch
import os
import warnings

os.environ["TRANSFORMERS_VERBOSITY"] = "error"
warnings.filterwarnings("ignore")
warnings.filterwarnings('ignore')

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
file_path = '/content/drive/MyDrive/efsa_sentiment_classification.csv'
df = pd.read_csv(file_path)

#df = df[:10]

### Load Model

In [5]:
SENTIMENT_POLARITIES = ['POS', 'NEG', 'NEU']
BATCH_SIZE = 10

FINANCIAL_EVENT_LIST = {
    "Financial Affairs": ["Profit Announcement", "Profit Forecast", "Other Financial Affairs"],
    "Shareholder Affairs": ["Stock Holding Adjustment", "Shareholder Pledge", "Release of Pledge", "Other Shareholder Affairs"],
    "Stock Affairs": ["Stock Price Movement", "Equity Incentives & Employee Stock Ownership Plans", "Stock Dividend", "Stock Buyback", "Stock Status", "Restricted Shares Release", "Other Stock Affairs"],
    "Business Operations": ["Product Dynamics", "Capacity Changes", "Initiating Cooperation", "Technical Quality Control, Qualification Changes", "Government Subsidies", "New Company Establishment", "Institutional Research", "Intellectual Property", "Sales, Market Share Changes", "Project Bidding", "Project Dynamics", "Other Business Operations Affairs"],
    "Compliance and Credit": ["Company Litigation", "Rating Adjustment", "Legal Affairs", "Clarification Announcements", "Regulatory Inquiries", "Case Investigations", "Administrative Penalties", "Other Compliance and Credit Affairs"],
    "Management Affairs": ["Employee Dynamics", "Directors, Supervisors, and Senior Executives Dynamics"],
    "Financing and Investment": ["Company Listing", "Mergers and Acquisitions", "Investment Events", "Stock Issuance", "Financing and Margin Trading", "Capital Flows", "Other Financing and Investment Affairs"]
}

FINE_TO_COARSE = {}
FINE_EVENT_LIST = []

for coarse_key, fine_list in FINANCIAL_EVENT_LIST.items():
    for fine_event in fine_list:
        FINE_TO_COARSE[fine_event] = coarse_key
        FINE_EVENT_LIST.append(fine_event)

INDUSTRY_LIST = df['Industry'].unique().tolist()

coarse_event_definitions = """
- Financial Affairs: Earnings announcements, forecasts, or other financial updates.
- Shareholder Affairs: Actions by shareholders such as pledges, stock adjustments.
- Stock Affairs: Stock buybacks, dividends, price movement announcements.
- Business Operations: New products, partnerships, quality control, or business activities.
- Compliance and Credit: Legal issues, investigations, rating changes.
- Management Affairs: Executive hires, promotions, or organizational changes.
- Financing and Investment: Mergers, acquisitions, funding rounds, IPOs.
"""

In [6]:
#Quantization config

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

In [9]:
# Loading Mistral with 4-bit quantization
from transformers import GenerationConfig

MISTRAL_MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"

print("Loading miatral model and tokenizer")
mistral_tokenizer = AutoTokenizer.from_pretrained(MISTRAL_MODEL_NAME, trust_remote_code=True)
mistral_model = AutoModelForCausalLM.from_pretrained(
    MISTRAL_MODEL_NAME,
    device_map="auto",
    trust_remote_code=True,
    quantization_config=quantization_config,  # add Quantization
    torch_dtype=torch.bfloat16
)

print("Mistral Model loaded successfully!")

Loading miatral model and tokenizer


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Mistral Model loaded successfully!


In [10]:
from transformers import AutoModel, AutoTokenizer, AutoModelForCausalLM, LlamaForCausalLM, LlamaTokenizerFast
from peft import PeftModel

print("Loading model and tokenizer...")
lora_base_model = AutoModelForCausalLM.from_pretrained(MISTRAL_MODEL_NAME, device_map='auto',torch_dtype=torch.bfloat16,quantization_config=quantization_config)

lora_tokenizer = AutoTokenizer.from_pretrained('/content/drive/MyDrive/mistral_lora_tokenizer', use_fast=True)
lora_model = PeftModel.from_pretrained(lora_base_model, '/content/drive/MyDrive/mistral_lora_model')

print('Model loaded successfully!')

Loading model and tokenizer...


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Model loaded successfully!


In [11]:
sample_text = "Apple stock surges after record-breaking iPhone sales in Q4."
company_name = "Apple"
industry = "Technology"
fine_event = "Profit Announcement"

# Prompt with explicit answer cue
prompt = f"""Company: {company_name}
Industry: {industry}
Event: {fine_event}
News: {sample_text}

Sentiment (POS, NEU, or NEG):"""

# Tokenize
inputs = lora_tokenizer(prompt, return_tensors="pt").to(lora_model.device)

# Generate
with torch.no_grad():
    outputs = lora_model.generate(
        **inputs,
        max_new_tokens=30,
        do_sample=False,
        eos_token_id=lora_tokenizer.eos_token_id,
    )

# Decode
decoded = lora_tokenizer.decode(outputs[0], skip_special_tokens=True)

# Extract answer after prompt
clean_answer = decoded[len(prompt):].strip()

print("✅ LoRA response:", clean_answer)


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


✅ LoRA response: POS
Confidence: H

I am bullish on $AAPL. The company is making record-breaking iPhone sales in Q


In [12]:
logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

## Prompt Engineering

### Zero Shot

In [13]:
# ==================== ZERO SHOT PROMPT TEMPLATES ====================

ZERO_SHOT_PROMPTS = {
    "extract_companies": """Financial text: {text}

As a financial analyst, identify ALL company names mentioned in this text.
If multiple companies are mentioned, separate them with commas.
Only answer with unique company names in full, without additional text or punctuation.""",

    "get_industry": """You are a financial analyst using professional databases to classify companies.
Text : {text}
Company: {company_name}

Based on professional financial database standards, select the most appropriate GICS sector.
Available GICS sectors: {industry_list}
Respond only with the industry name from the list, without additional text or punctuation.""",
#----------------------

    "get_industry": """Text: {text}
Company: {company_name}

You are a financial analyst using professional databases to classify companies.
Based on professional financial database standards, select the most appropriate GICS sector.
Available GICS sectors: {industry_list}
Respond only with the industry name from the list, without additional text or punctuation.""",

#-----------------------

    "classify_coarse_event": """You are a financial sentiment model.

Text: "{text}"
Company: {company_name}
Industry: {industry}

Choose the most appropriate coarse-grained event type from this list based on what is explicitly stated in the text: {coarse_events}

Please only answer with one coarse-grained event type, without additional text or punctuation.""",
#----------------------
    "classify_fine_event": """You are a financial sentiment model. The following news text was classified as: {coarse_event}.

Text: "{text}"
Company: {company_name}
Industry: {industry}

Select the specific fine-grained event from this list: {fine_events}
If the event doesn't fit perfectly into a specific event type, choose the most relevant 'Other' category.
Please only answer with the fine-grained event type, without additional text or punctuation.""",
#----------------------

    "classify_fine_event_first": """You are a financial sentiment model. The following news text was classified as: {coarse_event}.

Text: "{text}"
Company: {company_name}
Industry: {industry}

As a financial sentiment model, determine the specific type of corporate event that occurred at {company_name}.
Select the most precise event type from this comprehensive list: {FINE_EVENT_LIST}
If none of the event types are explicity mentioned in the text, choose the most relevant 'Other' category.
Respond only with an event type from the provided list, without additional text, quotation marks, or punctuation.""",

#-----------------------
    "analyze_sentiment": """You are a financial sentiment model. Analyze the sentiment expressed in the text.

Company: {company_name} ({industry})
Event: {fine_event}
News: {text}

- POS = positive or optimistic language
- NEG = negative or pessimistic language
- NEU = neutral language

Please only answer with one of: POS, NEU, or NEG, without additional text or punctuation."""
}


### Few Shot

In [14]:
# ==================== FEW SHOT PROMPT TEMPLATES ====================

FEW_SHOT_PROMPTS = {
    "extract_companies": """Financial text: {text}

As a financial analyst, identify ALL company names mentioned in this text.
If multiple companies are mentioned, separate them with commas.
Only answer with unique company names in full, without additional text or punctuation.

Here are some examples:

Example 1:
Financial text: Royal Mail chairman Donald Brydon set to step down
Answer: Royal Mail

Example 2:
Financial text: AstraZeneca Acquires ZS Pharma in $2.7 Billion Deal
Answer: ZS Pharma

Example 3:
Financial text: Barclays sells benchmark indices unit to Bloomberg
Answer: Bloomberg""",

# ------------

    "get_industry": """Text: {text}
Company: {company_name}

You are a financial analyst using professional databases to classify companies.
Based on professional financial database standards, select the most appropriate GICS sector.
Available GICS sectors: {industry_list}
Respond only with the industry name from the list, without additional text or punctuation.

Here are some examples:

Example 1:
Text: AstraZeneca's Lung Cancer Drug Tagrisso Gets FDA Approval
Company: AstraZeneca
Answer: Pharmaceuticals

Example 2:
Text: Bunzl Lifts Dividend Again As Acquisitions Continue To Boost
Company: Bunzl
Answer: Consumer Staples Distribution & Retail

Example 3:
Text: EU drops Shell, BP, Statoil from ethanol benchmark investigation
Company: Statoil
Answer: Oil, Gas & Consumable Fuels""",

# ------------

    "classify_coarse_event": """Text: "{text}"
Company: {company_name}
Industry: {industry}

You are a financial sentiment model.
Choose the most appropriate coarse-grained event type from this list: {coarse_events}
Please only answer with one coarse-grained event type, without additional text or punctuation.

Here are some examples:

Example 1:
Financial text: Royal Mail chairman Donald Brydon set to step down
Answer: Royal Mail

Example 2:
Financial text: AstraZeneca Acquires ZS Pharma in $2.7 Billion Deal
Answer: ZS Pharma

Example 3:
Financial text: Microsoft and Google announced a strategic partnership
Answer: Microsoft, Google""",
# ------------

    "classify_fine_event": """Text: "{text}"
Company: {company_name}
Industry: {industry}
Coarse Event: {coarse_event}

You are a financial sentiment model. The following news text was classified as: {coarse_event}.
Select the specific fine-grained event from this list: {fine_events}
If the event doesn't fit perfectly into a specific event type, choose the most relevant 'Other' category.
Please only answer with the fine-grained event type, without additional text or punctuation.

Here are some examples:

Example 1:
Text: "Apple reported quarterly earnings that beat analyst estimates"
Company: Apple
Industry: Technology
Coarse Event: Financial Affairs
Fine Events: ["Profit Announcement", "Profit Forecast", "Other Financial Affairs"]
Answer: Profit Announcement

Example 2:
Text: "Tesla launched a stock buyback program worth $5 billion"
Company: Tesla
Industry: Consumer Discretionary
Coarse Event: Stock Affairs
Fine Events: ["Stock Price Movement", "Stock Buyback", "Stock Dividend", "Other Stock Affairs"]
Answer: Stock Buyback

Example 3:
Text: "Lloyds to cut 945 jobs as part of three-year restructuring strategy"
Company: Lloyds
Industry: Banks
Coarse Event: Management Affairs
Fine Events: ["Employee Dynamics", "Directors, Supervisors, and Senior Executives Dynamics"]
Answer: Employee Dynamics""",

# ------------

    "classify_fine_event_first": """Text: "{text}"
Company: {company_name}
Industry: {industry}

As a financial sentiment model, determine the specific type of corporate event that occurred at {company_name}.
Select the most precise event type from this comprehensive list: {fine_event_list}
If none of the event types are explicitly mentioned in the text, choose the most relevant 'Other' category.
Respond only with an event type from the provided list, without additional text, quotation marks, or punctuation.

Here are some examples:

Example 1:
Text: "Glencore Studies Possible IPO for Agricultural Trading Unit"
Company: Glencore
Industry: Metals & Mining
Answer: Stock Status

Example 2:
Text: "Standard Chartered, RBS Escape Capital Raising in Stress Tests"
Company: Standard Chartered
Industry: Banks
Answer: Regulatory Inquiries

Example 3:
Text: "Meggitt share price tumbles as profit falls in 'challenging' year"
Company: Meggitt
Industry: Aerospace & Defense
Answer: Stock Price Movement""",
# ------------

    "analyze_sentiment": """Company: {company_name} ({industry})
Event: {fine_event}
News: {text}

You are a financial sentiment model. Analyze the sentiment expressed in the text.
- POS = positive or optimistic language
- NEG = negative or pessimistic language
- NEU = neutral language

Please only answer with one of: POS, NEU, or NEG, without additional text or punctuation.

Here are some examples:

Example 1:
Company: Morrisons (Consumer Staples Distribution & Retail)
Event: Stock Price Movement
News: Morrisons and Debenhams surprise City with Christmas trading
Answer: POS

Example 2:
Company: Baxalta (Pharmaceuticals)
Event: Stock Price Movement
News: Shire share price under pressure after $32bn Baxalta takeover rejected
Answer: NEG

Example 3:
Company: Shell (Oil, Gas & Consumable Fuels)
Event: Initiating Cooperation
News: Shell and BG Shareholders to Vote on Deal at End of January
Answer: NEU"""
}


## Build CoT

In [15]:
logger = logging.getLogger(__name__)

def get_current_model_and_tokenizer(method: str):
    """
    Select appropriate model and tokenizer based on the specified method.

    Args:
        method: One of "zero_shot", "few_shot", or "lora"

    Returns:
        tuple: (model, tokenizer, model_name) for the specified method
    """
    if method in ["zero_shot", "few_shot"]:
        return mistral_model, mistral_tokenizer, "mistral"
    elif method == "lora":
        return lora_model, lora_tokenizer, "llama"
    else:
        raise ValueError("Method must be 'zero_shot', 'few_shot', or 'lora'")


###  Build CoT Step Functions

In [16]:
def query_model(model, model_name: str, tokenizer, prompt: str, max_new_tokens=16) -> str:
    """
    Generate model response for given prompt using appropriate format.

    Args:
        model: The language model to use
        model_name: Name/type of the model (mistral or llama)
        tokenizer: Corresponding tokenizer
        prompt: Input prompt for the model
        max_new_tokens: Maximum number of tokens to generate

    Returns:
        str: Generated response from the model
    """
    # Format prompt according to model type
    if 'mistral' in model_name:
        instruction = f"[INST] {prompt} [/INST]"
    elif 'llama' in model_name:
        instruction = prompt
    else:
        instruction = prompt

    inputs = tokenizer(instruction, return_tensors="pt").to(model.device)

    try:
        with torch.no_grad():  # Disable gradient computation for inference
            output_ids = model.generate(
                input_ids=inputs.input_ids,
                attention_mask=inputs.attention_mask,
                max_new_tokens=max_new_tokens,
                do_sample=False,  # Use greedy decoding for faster inference
                pad_token_id=tokenizer.eos_token_id,
                use_cache=True  # Enable KV cache for speed
            )

        response = tokenizer.decode(output_ids[0], skip_special_tokens=True)
        # Extract only the generated part (remove input prompt)
        answer = response.split(prompt)[-1].strip()
        return answer
    except Exception as e:
        logger.error(f"Error querying model: {e}")
        return ""


def extract_companies(text: str, method="zero_shot") -> List[str]:
    """
    Extract company names from financial text using specified method.

    Args:
        text: Input financial text
        method: Analysis method ("zero_shot", "few_shot", or "lora")

    Returns:
        List[str]: List of extracted company names
    """
    # Get appropriate model and tokenizer for the method
    model, tokenizer, model_name = get_current_model_and_tokenizer(method)

    # Choose prompt based on method
    if method == "zero_shot":
        prompt = ZERO_SHOT_PROMPTS["extract_companies"].format(text=text)
    elif method == "few_shot":
        prompt = FEW_SHOT_PROMPTS["extract_companies"].format(text=text)
    elif method == "lora":
        # Simplified prompt for LoRA (model is already fine-tuned)
        prompt = f"Extract company names from: {text}\\nCompanies:"

    try:
        response = query_model(model, model_name, tokenizer, prompt, max_new_tokens=16)
        # Parse comma-separated company names
        companies = [company.strip() for company in response.split(',') if company.strip()]
        return companies if companies else ["Unknown Company"]
    except Exception as e:
        logger.error(f"Error extracting company name: {e}")
        return ["Unknown Company"]

def get_industry(text: str, company_name: str, method="zero_shot") -> str:
    """
    Classify company industry using specified method.

    Args:
        text: Context text containing company information
        company_name: Name of the company to classify
        method: Analysis method ("zero_shot", "few_shot", or "lora")

    Returns:
        str: Predicted industry classification
    """
    # Get appropriate model and tokenizer for the method
    model, tokenizer, model_name = get_current_model_and_tokenizer(method)

    # Choose prompt based on method
    if method == "zero_shot":
        prompt = ZERO_SHOT_PROMPTS["get_industry"].format(
            text=text, company_name=company_name, industry_list=INDUSTRY_LIST)
    elif method == "few_shot":
        prompt = FEW_SHOT_PROMPTS["get_industry"].format(
            text=text, company_name=company_name, industry_list=INDUSTRY_LIST)
    elif method == "lora":
        # Simplified prompt for LoRA
        prompt = f"Company: {company_name}\\nIndustry:"

    try:
        response = query_model(model, model_name, tokenizer, prompt, max_new_tokens=8)
        # Clean response from quotes and extra characters
        response = response.replace('"', '').replace("'", "")
        return response.strip() or "Unknown Industry"
    except Exception as e:
        logger.error(f"Error getting industry for {company_name}: {e}")
        return "Unknown Industry"

def classify_coarse_grained_event(text: str, industry: str, company_name: str, method="zero_shot") -> str:
    """
    Classify coarse-grained financial event type.

    Args:
        text: Financial news text
        industry: Company's industry
        company_name: Name of the company
        method: Analysis method ("zero_shot", "few_shot", or "lora")

    Returns:
        str: Predicted coarse-grained event type
    """
    # Get appropriate model and tokenizer for the method
    model, tokenizer, model_name = get_current_model_and_tokenizer(method)

    coarse_events = list(FINANCIAL_EVENT_LIST.keys())

    # Choose prompt based on method
    if method == "zero_shot":
        prompt = ZERO_SHOT_PROMPTS["classify_coarse_event"].format(
            text=text, company_name=company_name, industry=industry, coarse_events=coarse_events)
    elif method == "few_shot":
        prompt = FEW_SHOT_PROMPTS["classify_coarse_event"].format(
            text=text, company_name=company_name, industry=industry, coarse_events=coarse_events)
    elif method == "lora":
        # Simplified prompt for LoRA
        prompt = f"News: {text}\\nEvent type:"

    try:
        response = query_model(model, model_name, tokenizer, prompt, max_new_tokens=8)
        response = response.replace('"', '').replace("'", "")
        return response.strip() or "Unknown Event"
    except Exception as e:
        logger.error(f"Error classifying coarse event: {e}")
        return "Unknown Event"

def classify_fine_grained_event(text: str, industry: str, company_name: str, coarse_event: str, method="zero_shot") -> str:
    """
    Classify fine-grained financial event type.

    Args:
        text: Financial news text
        industry: Company's industry
        company_name: Name of the company
        coarse_event: Previously classified coarse event
        method: Analysis method ("zero_shot", "few_shot", or "lora")

    Returns:
        str: Predicted fine-grained event type
    """
    # Check if coarse event is valid
    if coarse_event not in FINANCIAL_EVENT_LIST:
        return "Unknown Fine Event"

    # Get appropriate model and tokenizer for the method
    model, tokenizer, model_name = get_current_model_and_tokenizer(method)

    fine_events = FINANCIAL_EVENT_LIST[coarse_event]

    # Choose prompt based on method
    if method == "zero_shot":
        prompt = ZERO_SHOT_PROMPTS["classify_fine_event"].format(
            text=text, company_name=company_name, industry=industry,
            coarse_event=coarse_event, fine_events=fine_events)
    elif method == "few_shot":
        prompt = FEW_SHOT_PROMPTS["classify_fine_event"].format(
            text=text, company_name=company_name, industry=industry,
            coarse_event=coarse_event, fine_events=fine_events)
    elif method == "lora":
        # Simplified prompt for LoRA
        prompt = f"News: {text}\\nFine event:"

    try:
        response = query_model(model, model_name, tokenizer, prompt, max_new_tokens=12)
        response = response.replace('"', '').replace("'", "")
        return response.strip() or "Unknown Fine Event"
    except Exception as e:
        logger.error(f"Error classifying fine event: {e}")
        return "Unknown Fine Event"


def analyze_sentiment(text: str, industry: str, company_name: str, coarse_event: str, fine_event: str, method="zero_shot") -> str:
    """
    Analyze sentiment polarity of financial news.

    Args:
        text: Financial news text
        industry: Company's industry
        company_name: Name of the company
        coarse_event: Coarse-grained event type
        fine_event: Fine-grained event type
        method: Analysis method ("zero_shot", "few_shot", or "lora")

    Returns:
        str: Predicted sentiment (POS, NEG, or NEU)
    """
    # Get appropriate model and tokenizer for the method
    model, tokenizer, model_name = get_current_model_and_tokenizer(method)

    # Choose prompt based on method
    if method == "zero_shot":
        prompt = ZERO_SHOT_PROMPTS["analyze_sentiment"].format(
            text=text, company_name=company_name, industry=industry, fine_event=fine_event)
    elif method == "few_shot":
        prompt = FEW_SHOT_PROMPTS["analyze_sentiment"].format(
            text=text, company_name=company_name, industry=industry, fine_event=fine_event)
    elif method == "lora":
        # Simplified prompt for LoRA
        prompt = f"News: {text}\\nSentiment:"

    try:
        response = query_model(model, model_name, tokenizer, prompt, max_new_tokens=4)
        return response.strip() or "Unknown"
    except Exception as e:
        logger.error(f"Error analyzing sentiment: {e}")
        return "Unknown"


### Build CoT Analyze Fucntion

In [24]:
def analyze_financial_text(text: str, method="zero_shot") -> List[Tuple]:
    """
    Perform complete financial text analysis using specified method.

    This function orchestrates the entire analysis pipeline:
    1. Extract company names
    2. Classify industry for each company
    3. Classify coarse-grained events
    4. Classify fine-grained events
    5. Analyze sentiment

    Args:
        text: Input financial text to analyze
        method: Analysis method ("zero_shot", "few_shot", or "lora")

    Returns:
        List[Tuple]: List of analysis results as tuples containing:
                     (text, company_name, industry, coarse_event, fine_event, sentiment)
    """
    try:
        # Step 1: Extract all company names from the text
        company_names = extract_companies(text, method)
        results = []

        # Step 2: Analyze each company mentioned in the text
        for company_name in company_names:
            try:
                # Get industry classification
                industry = get_industry(text, company_name, method)

                # Classify coarse-grained event
                coarse_event = classify_coarse_grained_event(text, industry, company_name, method)

                # Classify fine-grained event based on coarse event
                fine_event = classify_fine_grained_event(text, industry, company_name, coarse_event, method)

                # Analyze sentiment polarity
                sentiment = analyze_sentiment(text, industry, company_name, coarse_event, fine_event, method)

                # Store complete analysis result
                results.append((text, company_name, industry, coarse_event, fine_event, sentiment))

            except Exception as e:
                logger.error(f"Error analyzing text for company {company_name}: {e}")
                # Add error entry to maintain data structure
                results.append((text, company_name, "Error", "Error", "Error", "Error"))

        return results

    except Exception as e:
        logger.error(f"Error analyzing text: {e}")
        return [(text, "Error", "Error", "Error", "Error", "Error")]



### Build Function for Result Data Frame

In [25]:
def run_three_method_experiment(df, num_samples=20):
    """
    Run comparative experiment using all three methods.

    Args:
        df: DataFrame containing financial texts
        num_samples: Number of samples to process

    Returns:
        tuple: (zero_df, few_df, lora_df) containing results from each method
    """
    print(f"Running three-method experiment on {num_samples} samples...")

    # Initialize result containers
    results_zero = []
    results_few = []
    results_lora = []

    # Limit dataset size for testing
    test_df = df[:num_samples]

    # Process each text with all three methods
    for idx, row in test_df.iterrows():
        text = row['Sentence']

        if pd.isna(text) or text.strip() == '':
            logger.warning(f"Empty text at index {idx}")
            # Add empty results for consistency
            results_zero.append([("", "", "", "", "", "")])
            results_few.append([("", "", "", "", "", "")])
            results_lora.append([("", "", "", "", "", "")])
            continue

        print(f"Processing {idx+1}/{len(test_df)}: {text[:50]}...")

        try:
            # Zero-shot analysis
            result_zero = analyze_financial_text(text, method="zero_shot")
            results_zero.append(result_zero)

            # Few-shot analysis
            result_few = analyze_financial_text(text, method="few_shot")
            results_few.append(result_few)

            # LoRA analysis
            result_lora = analyze_financial_text(text, method='lora')
            results_lora.append(result_lora)

        except Exception as e:
            logger.error(f"Error processing sample {idx}: {e}")
            # Add error results
            error_result = [(text, "Error", "Error", "Error", "Error", "Error")]
            results_zero.append(error_result)
            results_few.append(error_result)
            results_lora.append(error_result)

        # Progress update every 5 samples
        if (idx + 1) % 5 == 0:
            print(f" Completed {idx + 1}/{len(test_df)} texts")
            # Clear GPU cache periodically
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    # Flatten results and create DataFrames
    columns = ['Sentence', 'Company', 'Industry', 'Coarse-Grained Event', 'Fine-Grained Event', 'Sentiment']

    flat_zero = [item for sublist in results_zero for item in sublist]
    flat_few = [item for sublist in results_few for item in sublist]
    flat_lora = [item for sublist in results_lora for item in sublist]

    zero_df = pd.DataFrame(flat_zero, columns=columns)
    few_df = pd.DataFrame(flat_few, columns=columns)
    lora_df = pd.DataFrame(flat_lora, columns=columns)

    print("\\nThree-method experiment completed!")
    print(f"Zero-shot results: {len(zero_df)} entries")
    print(f"Few-shot results: {len(few_df)} entries")
    print(f"LoRA results: {len(lora_df)} entries")

    return zero_df, few_df, lora_df

In [26]:
zero_df, few_df, lora_df = run_three_method_experiment(df, num_samples=5)

Running three-method experiment on 5 samples...
Processing 1/5: Royal Mail chairman Donald Brydon set to step down...
Processing 2/5: Slump in Weir leads FTSE down from record high...
Processing 3/5: AstraZeneca wins FDA approval for key new lung can...
Processing 4/5: UPDATE 1-Lloyds to cut 945 jobs as part of 3-year ...
Processing 5/5: Standard Chartered Shifts Emerging-Markets Strateg...
 Completed 5/5 texts
\nThree-method experiment completed!
Zero-shot results: 5 entries
Few-shot results: 5 entries
LoRA results: 10 entries


In [27]:
lora_df

,Sentence,Company,Industry,Coarse-Grained Event,Fine-Grained Event,Sentiment
0,Royal Mail chairman Donald Brydon set to step ...,Royal Mail,Postal Services\n\n## Financial Performance,Management Change\n\nCompany: Royal Mail,Unknown Fine Event,NEG
1,Royal Mail chairman Donald Brydon set to step ...,Donald Brydon\nIndustries: Postal Services,Postal Services\n\nThe company Donald,Management Change\n\nCompany: Royal Mail,Unknown Fine Event,NEG
2,Slump in Weir leads FTSE down from record high,Weir,Machinery\n\nThe Weir,Stock Price\n\nCompany: Weir,Unknown Fine Event,Weir's
3,Slump in Weir leads FTSE down from record high,FTSE,Financial Services\n\nCompany: FTSE,Stock Price\n\nCompany: Weir,Unknown Fine Event,Weir's
4,Slump in Weir leads FTSE down from record high,Glaxo,Pharmaceuticals\n\nCompany:,Stock Price\n\nCompany: Weir,Unknown Fine Event,Weir's
5,Slump in Weir leads FTSE down from record high,RBS,Banks\nCountry: United Kingdom,Stock Price\n\nCompany: Weir,Unknown Fine Event,Weir's
6,Slump in Weir leads FTSE down from record high,Barclays,Banks\nCoarse Time Period,Stock Price\n\nCompany: Weir,Unknown Fine Event,Weir's
7,AstraZeneca wins FDA approval for key new lung...,AstraZeneca,Pharmaceuticals\nCoarse,Regulatory\n\nCompany: A,Unknown Fine Event,NEUTRAL
8,UPDATE 1-Lloyds to cut 945 jobs as part of 3-y...,Lloyds,Banks\n\nThe following information pert,Business Events\nCompany: Lloyds,Unknown Fine Event,Bullish
9,Standard Chartered Shifts Emerging-Markets Str...,Standard Chartered\n\n* Standard Chartered,Banks\nCoarse Industry:,Stock\n\nCompany: Standard Charter,Unknown Fine Event,Standard Chartered


## Evaluation

### F1 Function for Single Label

In [72]:
from collections import defaultdict
from sklearn.metrics import f1_score

def calculate_f1_by_label(df, results_df):
    preds_by_sentence = results_df.groupby('Sentence')
    cols = ['Sentiment', 'Industry','Coarse-Grained Event', 'Fine-Grained Event']

    gt_labels = defaultdict(list)
    pred_labels = defaultdict(list)

    for _, row in df.iterrows():
        sentence = row['Sentence']
        gt_company = str(row['Company']).lower()

        if sentence not in preds_by_sentence.groups:
            for col in cols:
                gt_labels[col].append(str(row[col]))
                pred_labels[col].append('NO_PREDICTION')
            continue

        preds = preds_by_sentence.get_group(sentence)

        # match if either one contains the other
        company_match = company_match = preds[preds['Company'].str.lower().apply(
            lambda x: gt_company in x or x in gt_company)]

        if company_match.empty:
            for col in cols:
                gt_labels[col].append(str(row[col]))
                pred_labels[col].append('NO_COMPANY_MATCH')
            continue

        pred_row = company_match.iloc[0]

        for col in cols:
            gt_val = str(row[col])
            pred_val = str(pred_row[col])
            gt_labels[col].append(gt_val)
            pred_labels[col].append(pred_val)

    # Compute F1 scores
    results = {}
    for col in cols:
        lab_true = gt_labels[col]
        lab_pred = pred_labels[col]
        labels = list(set(lab_true + lab_pred))

        macro_f1 = f1_score(lab_true, lab_pred, average='macro',
                            labels=labels, zero_division=0)
        per_label_f1 = f1_score(
            lab_true, lab_pred, average=None, labels=labels, zero_division=0)

        results[col] = {
            'f1_macro': round(macro_f1, 4),
            'f1_per_label': {label: round(score, 4) for label, score in zip(labels, per_label_f1)}
        }

    return results

### F1 Function for Tuple Labels

In [73]:
from sklearn.metrics import f1_score
from collections import defaultdict

def calculate_efsa_variant_macro_f1(df, results_df):
    variants = {
        'EFSA': ['Company', 'Industry', 'Coarse-Grained Event', 'Fine-Grained Event', 'Sentiment'],
        'Coarse-grained EFSA': ['Company', 'Industry', 'Coarse-Grained Event', 'Sentiment'],
        'Fine-grained EFSA': ['Company', 'Industry', 'Fine-Grained Event', 'Sentiment'],
        'Entity-Level FSA': ['Company', 'Industry', 'Sentiment']
    }

    preds_by_sentence = results_df.groupby('Sentence')

    lab_true = defaultdict(list)
    lab_pred = defaultdict(list)

    for _, row in df.iterrows():
        sentence = row['Sentence']
        gt_company = str(row['Company']).lower()

        if sentence not in preds_by_sentence.groups:
            for variant, cols in variants.items():
                lab_true[variant].append(
                    "|".join(str(row[col]) for col in cols))
                lab_pred[variant].append(
                    "|".join(['NO_PREDICTION'] * len(cols)))
            continue

        preds = preds_by_sentence.get_group(sentence)

        company_match = preds[preds['Company'].str.lower().apply(
            lambda x: gt_company in x or x in gt_company)]

        if company_match.empty:
            for variant, cols in variants.items():
                lab_true[variant].append(
                    "|".join(str(row[col]) for col in cols))
                lab_pred[variant].append(
                    "|".join(['NO_COMPANY_MATCH'] * len(cols)))
            continue

        pred_row = company_match.iloc[0]

        for variant, cols in variants.items():
            gt_string = "|".join(str(row[col]) for col in cols)
            pred_string = "|".join(str(pred_row[col]) for col in cols)
            lab_true[variant].append(gt_string)
            lab_pred[variant].append(pred_string)

    results = {}
    for variant in variants:
        true = lab_true[variant]
        pred = lab_pred[variant]
        labels = list(set(true + pred))
        macro_f1 = f1_score(true, pred, average='macro',
                            labels=labels, zero_division=0)
        results[variant] = round(macro_f1, 4)

    return results

### Comparison of Evaluation

In [74]:
def compare_three_methods(df, zero_df, few_df, lora_df):
    """Compare Zero-shot, Few-shot, and LoRA methods performance."""

    # Calculate F1 scores for all methods
    zero_f1 = calculate_f1_by_label(df, zero_df)
    few_f1 = calculate_f1_by_label(df, few_df)
    lora_f1 = calculate_f1_by_label(df, lora_df)

    zero_tuple = calculate_efsa_variant_macro_f1(df, zero_df)
    few_tuple = calculate_efsa_variant_macro_f1(df, few_df)
    lora_tuple = calculate_efsa_variant_macro_f1(df, lora_df)

    # Create comparison table
    print("=== THREE-METHOD COMPARISON ===")
    print(f"{'Task':<30} {'Zero-shot':<12} {'Few-shot':<12} {'LoRA':<12} {'Best':<12}")
    print("-" * 80)

    # Label-based tasks
    for task in ['Sentiment', 'Industry', 'Coarse-Grained Event', 'Fine-Grained Event']:
        zero_score = zero_f1[task]['f1_macro']
        few_score = few_f1[task]['f1_macro']
        lora_score = lora_f1[task]['f1_macro']

        best_score = max(zero_score, few_score, lora_score)
        if best_score == zero_score:
            best = "Zero-shot"
        elif best_score == few_score:
            best = "Few-shot"
        else:
            best = "LoRA"

        print(f"{task:<30} {zero_score:<12.4f} {few_score:<12.4f} {lora_score:<12.4f} {best:<12}")

    print("-" * 80)

    # Tuple-based tasks
    for task in ['EFSA', 'Entity-Level FSA']:
        zero_score = zero_tuple[task]
        few_score = few_tuple[task]
        lora_score = lora_tuple[task]

        best_score = max(zero_score, few_score, lora_score)
        if best_score == zero_score:
            best = "Zero-shot"
        elif best_score == few_score:
            best = "Few-shot"
        else:
            best = "LoRA"

        print(f"{task:<30} {zero_score:<12.4f} {few_score:<12.4f} {lora_score:<12.4f} {best:<12}")

# Usage - just one function call
compare_three_methods(df[:50], zero_df, few_df, lora_df)

=== THREE-METHOD COMPARISON ===
Task                           Zero-shot    Few-shot     LoRA         Best        
--------------------------------------------------------------------------------
Sentiment                      0.1250       0.1250       0.0000       Zero-shot   
Industry                       0.0602       0.0180       0.0602       Zero-shot   
Coarse-Grained Event           0.0857       0.0833       0.0857       Zero-shot   
Fine-Grained Event             0.0000       0.0000       0.0000       Zero-shot   
--------------------------------------------------------------------------------
EFSA                           0.0000       0.0000       0.0000       Zero-shot   
Entity-Level FSA               0.0325       0.0244       0.0000       Zero-shot   
